# فصل ۱۰: حافظه و نگهداری تاریخچه (Memory & History)
## بخش ۴: حافظه خلاصه‌ساز (Summary Memory)

### چرا حافظه خلاصه‌ساز؟
در درسنامه قبل با **هرس پیام‌ها (`trim_messages`)** آشنا شدیم، اما مشکل اساسی هرس کردن این بود که اطلاعات کلیدی قدیمی (مثل نام کاربر یا قوانین اولیه بازی) پس از چند پیام به کلی فراموش می‌شدند.

در تکنیک **حافظه خلاصه‌ساز (Summary Memory)** به جای دور ریختن پیام‌های قدیمی:
1. تاریخچه مکالمه به یک **زنجیره خلاصه‌ساز** سپرده می‌شود تا نکات کلیدی آن را فشرده و استخراج کند.
2. خلاصه تولیدشده در قالب **`SystemMessage`** (و نه `AIMessage`!) به زنجیره اصلی چت ارسال می‌شود.
3. زنجیره اصلی بر اساس این خلاصه و سوال جدید کاربر پاسخ دقیق تولید می‌کند.

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# بارگذاری متغیرهای محیطی
load_dotenv()

# مدل اصلی برای پاسخ‌گویی به کاربر
chat_model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.7
)

# مدل خلاصه‌ساز (در پروژه‌های واقعی می‌تواند یک مدل ارزان‌تر و سریع‌تر با دمای پایین باشد)
summary_model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0
)

print("✅ مدل اصلی و مدل خلاصه‌ساز آماده شدند!")

✅ مدل اصلی و مدل خلاصه‌ساز آماده شدند!


### ۱. تعریف تابع خلاصه‌ساز (`summarize_history`)

این تابع لیست پیام‌های تاریخچه را می‌گیرد؛ اگر تعداد پیام‌ها کمتر از ۲ باشد نیازی به خلاصه‌سازی نیست، در غیر این صورت متن گفتگو را به یک رشته واحد تبدیل کرده و به مدل خلاصه‌ساز می‌دهد.

In [2]:
def summarize_history(history_messages):
    """خلاصه‌سازی پیام‌های تاریخچه و استخراج نکات کلیدی"""
    if not history_messages or len(history_messages) < 2:
        return ""

    summary_prompt = ChatPromptTemplate.from_messages([
        ("system", "تو یک دستیار هوشمند هستی. وظیفه تو این است که مکالمات زیر را خلاصه کنی و نکات کلیدی آن (مانند نام کاربر، علایق، شغل و موضوعات مهم) را استخراج کنی."),
        ("user", "مکالمات:\n{conversation}\n\nخلاصه فشرده:"),
    ])

    # استفاده از msg.text برای استخراج تمیز متن از پیام‌های ذخیره شده
    conversation_text = "\n".join([f"{msg.type}: {msg.text}" for msg in history_messages])

    summary_chain = summary_prompt | summary_model
    summary = summary_chain.invoke({"conversation": conversation_text})
    
    return summary.text

print("✅ تابع summarize_history تعریف شد!")

✅ تابع summarize_history تعریف شد!


### ۲. تعریف مخزن تاریخچه (`store`) و ساخت زنجیره اصلی

> **نکته بسیار مهم:** خلاصه گفت‌وگو حتماً باید در قالب **`SystemMessage`** به مدل اصلی تزریق شود، نه به عنوان `AIMessage`. تزریق خلاصه به شکل `AIMessage` مدل را گمراه می‌کند که گویا خودش قبلاً آن متن را نوشته و ربطی به اطلاعات کاربر ندارد.

In [3]:
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "تو یک دستیار مفید هستی. کوتاه و دقیق پاسخ بده. خلاصه مکالمات قبلی:\n{summary}"),
    ("human", "{input}"),
])

chain = (
    RunnablePassthrough.assign(
        summary=lambda x: summarize_history(x["chat_history"])
    )
    | chat_prompt
    | chat_model
)

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("✅ خط لوله حافظه خلاصه‌ساز آماده استفاده است!")

✅ خط لوله حافظه خلاصه‌ساز آماده استفاده است!


d:\Project\LLM\llm-apps-course\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### ۳. اجرای چت تعاملی با استریمینگ زنده (Streaming)

**سناریوی پیشنهادی برای تست حافظه خلاصه‌ساز:**
1. `سلام. من طاها هستم و شغلم برنامه‌نویس است.`
2. `من به خواندن تاریخ ایران علاقه دارم. آخرین شاه قاجار در چه سنی به سلطنت رسید؟`
3. `پدر احمدشاه قاجار چه کسی بود؟`
4. `حالا بهم بگو شغل اصلی من چی بود؟`
5. `اسم من چی بود؟`
6. برای خروج `0` را وارد کنید.

In [4]:
print("--- شروع چت (برای خروج 0 را بزنید) ---")

config = {"configurable": {"session_id": "1234"}}

while True:
    user_input = input("User: ")
    if user_input.strip() == "0":
        print("پایان مکالمه.")
        break
    print(f"User: {user_input}")
    
    print("AI: ", end="", flush=True)
    ai_response = ""
    for chunk in with_message_history.stream(
        {"input": user_input},
        config=config
    ):
        text = chunk.text
        ai_response += text
        print(text, end="", flush=True)
    print()

--- شروع چت (برای خروج 0 را بزنید) ---
User: ثلام اثم من علیه
AI: 

C:\Users\aline\AppData\Local\Temp\ipykernel_2616\3191698065.py:5: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


سلام علی جان، خوشبختم. چطور می‌توانم کمکت کنم؟
User: من اثم هر باشگاه فوتبال رو که گفتم بهم بگو اثم باشگاه دیگر در همون شهر چیه اگر نداره یا نمیدونم بگو 
AI: سلام علی جان. متوجه شدم. نام باشگاهی که مد نظرت هست رو بگو تا رقیب همشهری‌اش رو بهت بگم. آماده‌ام!
User: رعال مادرید
AI: رقیب اصلی و همشهری رئال مادرید، باشگاه **اتلتیکو مادرید** است (دربی مادرید). 

همچنین بارسلونا رقیب سنتی رئال در سطح کشور است (ال‌کلاسیکو).
User: بارثلونا
AI: بارسلونا (FC Barcelona) یکی از بزرگ‌ترین و پر افتخارترین باشگاه‌های فوتبال جهان، اهل شهر بارسلون در ایالت کاتالونیای اسپانیا است. 

بزرگ‌ترین رقیب سنتی این تیم، **رئال مادرید** است که مسابقه بین آن‌ها به **«ال کلاسیکو»** معروف است و یکی از جذاب‌ترین رقابت‌های دنیای ورزش محسوب می‌شود. 

علی جان، دوست داری درباره تاریخچه، افتخارات یا رقیب همشهری بارسلونا (اسپانیول) بیشتر بدانی؟
User: از چت فاصله نگیر و دقیقا همون کاری رو که بهت گفتم رو بگو نه یک کلمه بیشتر نه یک کلمه کمتر
AI: اتلتیکو مادرید
User: بایرن مونیخ
AI: بوروسیا دورتموند
User: تاتنهام
AI: آرسنال.
Use

### ۴. مشاهده خلاصه نهایی تولیدشده از گفتگو

In [5]:
if "1234" in store:
    print("📝 خلاصه استخراج‌شده از کل مکالمه:")
    print("-" * 50)
    print(summarize_history(store["1234"].messages))
else:
    print("هنوز گفتگویی ثبت نشده است.")

📝 خلاصه استخراج‌شده از کل مکالمه:
--------------------------------------------------
**خلاصه مکالمه:**
کاربر در این مکالمه از هوش مصنوعی می‌خواهد که وقتی نام یک باشگاه فوتبال را می‌گوید، فقط نام باشگاه رقیب همشهری (و در صورت نیاز مرتبط) را بدون هیچ توضیح اضافی اعلام کند. کاربر پس از دریافت پاسخ‌های طولانی در ابتدا، ربات را متوجه دستور کار می‌کند و ربات ادامه پاسخ‌ها را بسیار کوتاه ارائه می‌دهد.

**نکات کلیدی:**
* **نام کاربر:** علی (دارای اختلال در تلفظ حروف در نگارش، مانند ث به جای س)
* **علایق:** فوتبال، باشگاه‌های فوتبال اروپایی
* **شغل:** ذکر نشده است.
* **موضوعات مهم:** درخواست پاسخ‌های بسیار کوتاه و دقیق درباره رقبای همشهری باشگاه‌های فوتبال (رئال مادرید، بارسلونا، بایرن مونیخ، تاتنهام، لیورپول، منچستر یونایتد و یوونتوس).


### 📌 جمع‌بندی و نکته بهینه‌سازی

- **مزیت:** مدل حتی در مکالمات بسیار طولانی اطلاعات کلیدی را فراموش نمی‌کند و Context Window هم پر نمی‌شود.
- **هزینه:** در این پیاده‌سازی ساده، به ازای هر پیام کاربر، مدل **دو بار** صدا زده می‌شود (یک بار برای خلاصه‌سازی و یک بار برای پاسخ).
- **راهکار در پروژه‌های واقعی:** معمولاً خلاصه‌سازی به صورت دوره‌ای (مثلاً پس از هر ۱۰ پیام جدید) یا در پس‌زمینه اجرا می‌شود و در کنار چند پیام اخیر (Sliding Window) به مدل ارسال می‌گردد.